### Indicators
- Revenue Growth (QoQ)
- EPS - Earnings Per Share
- Gross Margin - (Revenue - Cost of Goods) / Revenue
- Net Profit Margin - Percentage of revenue left after expenses
- ROE - Return on Equity, How effectively capital is being used
- ROIC - Return on Invested Capital, how well it turns all capital into profit
- ROA - Return on Assets
- Equity Multiplier (Total Assets / Stockholder's Equity)
- OCF - Operating cash flow
- CapEx - Expenditures
- FCF - (OCF - CapEx) Free Cash Flow, hard cash
- Cash King Ratio - (OCF / Net Income)
- D/E - Debt to Equity 
- EBIT (Earnings Before Interest & Taxes)
- EBITDA - EBIT + Deprecation and Ammortization (Cash generated by operations)
- Net Debt / EBITDA
- EV - Enterprise Value, EV = Market Cap + Total Debt - Cash and Equivalents
- EV/EBITDA 
- P/E - Price / Earnings
- PEG Ration (P/E) / (EPS growth rate)
- P/B - Price-to-Book, compares market value and assets
- EV/Revenue
- Current Ration (Current Assets / Current Liabilities)
- Net Debt (Total Debt - Cash) if we have more cash than debt
- Dividend Yield  (Annual dividend / Price)
- Dividend Payout Ratio: (Dividends / Net Income)
- Altman Z-Score (Z = 1.2A + 1.4B + 3.3C + 0.6D + 1.0E): 
    - A = Working Capital / Total Assets (Liquidity)
    - B = Retained Earnings / Total Assets (Accumulated Profitability)
    - C = EBIT / Total Assets (Current Profitability)
    - D = Market Value of Equity / Total Liabilities (Solvency)
    - E = Sales (Revenue) / Total Assets (Asset Efficiency)
- Sloan Ratio (Net Income - OCF) / Total Assets
- Owner Earnings (Net Income + Depreciation/Amortization – Maintenance CapEx)
- Rule of 40 (Revenue Growth % + FCF Margin %) (FCF Margin % = FCF / Revenue * 100)


In [7]:
import os
import requests
import json
from tqdm.notebook import tqdm
from datetime import date
import time

from edgar import *
import yfinance as yf
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
IDENTITY = os.getenv("IDENTITY")

COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_HEADERS = {
    "User-Agent": f"financial-research-masters-degree ({IDENTITY})",
    "Accept": "application/json"
}

EDGAR_DATA = "edgar_data"

MAX_BACKFILL_DAYS = 365 * 3

set_identity(IDENTITY)

RENAME_TABLE = {
    'revenue': 'totalRevenue',
    'operating_income': 'operatingIncome',
    'net_income': 'netIncome',
    'total_assets': 'totalAssets',
    'total_liabilities': 'totalLiabilities',
    'stockholders_equity': 'totalShareholderEquity',
    'current_assets': 'totalCurrentAssets',
    'current_liabilities': 'totalCurrentLiabilities',
    'operating_cash_flow': 'operatingCashflow',
    'capital_expenditures': 'capitalExpenditures',
    'free_cash_flow': 'freeCashFlow',
    'shares_outstanding_basic': 'commonStockSharesOutstanding',
    'shares_outstanding_diluted': 'commonStockSharesOutstandingDiluted'
}

METRICS = {
    'totalRevenue': [
        "Revenues", 
        "SalesRevenueNet", 
        "SalesRevenueGoodsNet", 
        "RevenuesNetOfInterestExpense",
    ],
    'operatingIncome': [
        "OperatingIncomeLoss",
        "OperatingIncomeLossFromContinuingOperations",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
        "IncomeLossFromContinuingOperationsBeforeInterestExpenseInterestIncomeIncomeTaxesExtraordinaryItemsNoncontrollingInterestsNet",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments",
    ],
    'operatingExpenses': ["OperatingExpenses", "CostsAndExpenses", "LeaseCost"],
    'nonoperatingIncome': [
        "NonoperatingIncomeExpense",
        "IncomeLossFromNonoperatingActivities",
        "OtherNonoperatingIncomeExpense",
        "OtherNonoperatingIncome",
    ],
    'netIncome': [
        "NetIncomeLoss",
        "ProfitLoss",
        "NetIncomeLossAvailableToCommonStockholdersBasic",
        "IncomeLossFromContinuingOperations",
    ],
    'totalAssets': ["Assets"],
    'totalLiabilities': ["Liabilities"],
    'totalShareholderEquity': ["StockholdersEquity"],
    'totalCurrentAssets': ["AssetsCurrent"],
    'nonCurrentAssets': ["NoncurrentAssets"],
    'totalCurrentLiabilities': ["LiabilitiesCurrent"],
    'nonCurrentLiabilities': ["LiabilitiesNoncurrent"],
    'liabilitiesAndStockholdersEquity': ["LiabilitiesAndStockholdersEquity"],
    'operatingCashflow': [
        "NetCashProvidedByUsedInOperatingActivities",
        "NetCashProvidedByUsedInContinuingOperations",
        "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
        ],
    'capitalExpenditures': [
        "PaymentsToAcquirePropertyPlantAndEquipment", 
        "CapitalExpendituresIncurredButNotYetPaid",
        "PaymentsToAcquireProductiveAssets",
        "PaymentsForCapitalImprovements",
    ],
    
    'grossProfit': ["GrossProfit"],
    'costofGoodsAndServicesSold': [
        "CostOfGoodsAndServicesSold",
        "CostofGoodsAndServicesSold",
        # "CostsAndExpenses",
        "CostOfGoodsSold", 
        "CostOfRevenue", 
        "CostOfSales", 
        "CostOfServices",
        "CostOfPurchasedOilAndGas",
    ],
    'depreciation': ["Depreciation"],
    'amortization': ["AmortizationOfIntangibleAssets", "FinanceLeaseRightOfUseAssetAmortization"],
    'depreciationAndAmortization': [
        "DepreciationDepletionAndAmortization", 
        "DepreciationAndAmortization",
        "DepreciationAmortizationAndAccretionNet",
        "DepreciationDepletionAndAmortizationIncludingDiscontinuedOperationDepreciationandAmortization",
        "DepreciationDepletionAndAmortizationincludingdiscontinuedoperations",
        "DepreciationAndAmortizationIncludingDiscontinuedOperations"
        ],
    # 'costOfGoodsSoldDepreciationAndAmortization': ["CostOfGoodsSoldDepreciationAndAmortization"],
    'cashAndCashEquivalentsAtCarryingValue': [
        "cashAndCashEquivalentsAtCarryingValue", 
        "CashAndCashEquivalentsAtCarryingValue",
        "CashCashEquivalentsAndShortTermInvestments", 
        "CashAndCashEquivalentsPeriodIncreaseDecrease", 
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
        "CashAndDueFromBanks",
    ],
    'longTermInvestments': [ #TODO: Make use of it
        "LongTermInvestments",
        "AvailableForSaleSecuritiesNoncurrent",
        "InvestmentsAndAdvancesSecurities",
        "OtherInvestmentsNoncurrent"
    ],
    'shortTermDebt': [
        "DebtCurrent",                          # The "Holy Grail" - total current debt
        "ShortTermBorrowings",                  # Very common consolidated tag
        "LongTermDebtCurrent",                  # The portion of big loans due THIS year
        "LongTermDebtAndCapitalLeaseObligationsCurrent",
        "CommercialPaper",                      # Vital for Banks/Blue Chips
        "NotesPayableCurrent",                  # Standard for smaller loans
        "LinesOfCreditCurrent",                 # Revolver draws
        "ConvertibleDebtCurrent",               # Tech/Biotech specific
        "NotesAndLoansPayableCurrent",          # General bank debt
        "OtherShortTermBorrowings",             # The catch-all
        "SecuredDebtCurrent",                   # Specific type
        "UnsecuredShortTermBorrowings"          # Common for financials
    ],
    'longTermDebt': [
        "LongTermDebt", 
        "LongTermDebtAndCapitalLeaseObligations", 
        "LongTermDebtNoncurrent",
        "LongTermNotesPayable",
        "ConvertibleLongTermNotesPayable",
        "SeniorLongTermNotes",
        "UnsecuredLongTermDebt"
        ],
    'retainedEarnings': ["RetainedEarningsAccumulatedDeficit"],
    'dividendPayout': [
        "PaymentsOfDividends", 
        "PaymentsOfDividendsCommonStock",
        "DividendsCommonStockCash",
        "PaymentsOfOrdinaryDividends",
    ],
    'commonStockDividendsPerShareCashPaid': ['CommonStockDividendsPerShareCashPaid'],
    'interestExpense': [
        "InterestExpense",                           # Total (Best)
        "InterestAndDebtExpense",                    # Combined
        "InterestExpenseNonoperating",               # Specific to non-bank debt
        "InterestExpenseOperating",                  # Specific to banks/lending
        "InterestAndFeeExpensesOperating",           # Common for GS
    ],
    'incomeTaxExpense': [
        "IncomeTaxExpenseBenefit", 
    ],
    'currentIncomeTaxExpense': [
        "CurrentIncomeTaxExpenseBenefit",
    ],
    'deferredIncomeTaxExpense': [
        "DeferredIncomeTaxExpenseBenefit",
    ],
    'ebitda': [  # some companies report directly
        "EarningsBeforeInterestTaxesDepreciationAndAmortization",
    ],
    'researchAndDevelopment': [
        "ResearchAndDevelopmentExpense",
        "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
        "ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost",
        "ResearchAndDevelopmentAssetAcquiredOtherThanThroughBusinessCombinationWrittenOff",
    ],
    'sellingGeneralAndAdministrative': [
        "SellingGeneralAndAdministrativeExpense",
        "GeneralAndAdministrativeExpense",
        "SellingAndMarketingExpense",
    ],
    'stockBasedCompensation': [
        "ShareBasedCompensation",
        "AllocatedShareBasedCompensationExpense",
        "EmployeeBenefitsAndShareBasedCompensation",
    ],
    'goodwill': [
        "Goodwill",
    ],
    'intangibleAssets': [
        "IntangibleAssetsNetExcludingGoodwill",
        "FiniteLivedIntangibleAssetsNet",
        "IndefiniteLivedIntangibleAssetsExcludingGoodwill",
    ],
    'accountsReceivable': [
        "AccountsReceivableNetCurrent",
        "ReceivablesNetCurrent",
        "AccountsAndNotesReceivableNet",
        "AccountsReceivableNet",
        "IncreaseDecreaseInAccountsReceivable",
        "IncreaseDecreaseInReceivables",
    ],
    'inventory': [
        "InventoryNet",
        "InventoryGross",
        "FIFOInventoryAmount",
        "RetailRelatedInventory",
    ],
    'accountsPayable': [
        "AccountsPayableCurrent",
        "AccountsPayableAndAccruedLiabilitiesCurrent",
        "IncreaseDecreaseInAccountsPayable",
    ],
    'accruedLiabilities': [
        "AccruedLiabilitiesCurrent",
        "EmployeeRelatedLiabilitiesCurrent",
        "OtherLiabilitiesCurrent",
    ],
    'deferredRevenue': [
        "DeferredRevenueCurrent",
        "DeferredRevenueNoncurrent",
        "ContractWithCustomerLiabilityCurrent",
        "ContractWithCustomerLiabilityNoncurrent",
    ],
    'propertyPlantEquipmentNet': [
        "PropertyPlantAndEquipmentNet",
        "PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization",
    ],
    'commonStockSharesOutstanding': [
        "WeightedAverageNumberOfSharesOutstandingBasic",
        "CommonStockSharesOutstanding",
        "EntityCommonStockSharesOutstanding",
        "SharesOutstanding",
    ],
    'commonStockSharesOutstandingDiluted': [
        "WeightedAverageNumberOfDilutedSharesOutstanding",
        "WeightedAverageNumberOfSharesOutstandingDiluted",
        "WeightedAverageNumberOfShareOutstandingDiluted",
        ],
    'commonStockSharesOutstandingBasicAndDiluted': ["WeightedAverageNumberOfShareOutstandingBasicAndDiluted"],
    'sharesBuyback': [
        "PaymentsForRepurchaseOfCommonStock",
        "StockRepurchasedAndRetiredDuringPeriodValue",
        "TreasuryStockValueAcquiredCostMethod",
        "StockRepurchasedDuringPeriodValue",
    ],
    'operatingLeaseRightOfUseAsset': [
        "OperatingLeaseRightOfUseAsset",
        "FinanceLeaseRightOfUseAsset",
    ],
    'operatingLeaseLiability': [
        "OperatingLeaseLiability",
        "OperatingLeaseLiabilityCurrent",
        "OperatingLeaseLiabilityNoncurrent",
    ],
    'noncontrollingInterest': [
        "MinorityInterest",
        "MinorityInterestInSubsidiaries",
    ],
    'comprehensiveIncome': [
        "ComprehensiveIncomeNetOfTax",
        "OtherComprehensiveIncomeLossNetOfTax",
    ],
    # Financing flows — useful for FCF yield, leverage analysis
    'proceedsFromDebt': [
        "ProceedsFromIssuanceOfLongTermDebt",
        "ProceedsFromIssuanceOfDebt",
        "ProceedsFromLinesOfCredit",
        "ProceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet",
        "ProceedsFromLoans",
        "ProceedsFromRepaymentsOfShortTermDebtMaturingInMoreThanThreeMonths_Duration",
    ],
    'repaymentsOfDebt': [
        "RepaymentsOfDebt",
        "RepaymentsOfLongTermDebt",
        "RepaymentsOfLinesOfCredit",
        "RepaymentsOfLongTermDebtAndCapitalSecurities",
        "RepaymentsOfCommercialPaper",
    ],
    'proceedsFromEquityIssuance': [
        "ProceedsFromIssuanceOfCommonStock",
        "ProceedsFromStockOptionsExercised",
    ],
}

In [ ]:
def get_company_metadata(ticker: str) -> dict:
    try:
        info = yf.Ticker(ticker).info
        return {
            "sector": info.get("sector", None),
            "industry": info.get("industry", None),
        }
    except Exception as e:
        print(f"Failed to get metadata for {ticker}: {e}")
        return {"sector": None, "industry": None, "marketCap": None}


def get_price_history(ticker: str) -> pd.Series | None:
    try:
        hist = yf.Ticker(ticker).history(period='max')
        hist.index = hist.index.tz_localize(None)
        return hist
    except Exception as e:
        print(f"Failed to get price history for {ticker}: {e}")
        return None
    
def lookup_price(price_history: pd.Series, date_str: str, window_days: int = 5) -> float | None:
    if not date_str or date_str == "None":
        return None
    
    try:
        target = pd.Timestamp(date_str)
        end = target + pd.Timedelta(days=window_days)
        subset = price_history[target:end]
        if subset.empty:
            return None
        return round(subset['Close'].iloc[0], 4)
    except Exception as e:
        print(f"Failed to collect subset of price history for date {date_str}: {e}")
        return None

def get_precise_quarterly_value(df, concept_tag, report_date):
    report_date_ts = pd.to_datetime(report_date)

    base = df[
        (df['clean_concept'] == concept_tag) &
        (df['dimension'].isna())
    ]
    if base.empty:
        return None, None

    # Instant values (balance sheet)
    # instants = base[base['period_start'].isna()]
    instants = base[
        (base['period_start'].isna()) &
        (base['period_end_ts'].notna()) &
        ((base['period_end_ts'] - report_date_ts).dt.days.abs() <= 7)
    ]
    if not instants.empty:
        return instants.iloc[0]['value'], instants.iloc[0]['source_filing_date']

    date_diff = (base['period_end_ts'] - report_date_ts).dt.days.abs()
    matches = base[date_diff <= 7].copy()
    if matches.empty:
        return None, None

    quarterly = matches[(matches['days'] > 80) & (matches['days'] < 115)]
    if not quarterly.empty:
        quarterly = quarterly.copy()
        quarterly['dist'] = (quarterly['days'] - 91).abs()
        val = quarterly.sort_values('dist').iloc[0]
        return val['value'], val['source_filing_date']

    return None, None


def get_precise_annual_value(df, concept_tag, report_date):
    report_date_ts = pd.to_datetime(report_date)

    base = df[
        (df['clean_concept'] == concept_tag) &
        (df['dimension'].isna())
    ]
    if base.empty:
        return None, None

    # instants = base[base['period_start'].isna()]
    instants = base[
        (base['period_start'].isna()) &
        (base['period_end_ts'].notna()) &
        ((base['period_end_ts'] - report_date_ts).dt.days.abs() <= 7)
    ]
    if not instants.empty:
        return instants.iloc[0]['value'], instants.iloc[0]['source_filing_date']

    date_diff = (base['period_end_ts'] - report_date_ts).dt.days.abs()
    matches = base[date_diff <= 7].copy()
    if matches.empty:
        return None, None

    annual = matches[(matches['days'] > 350) & (matches['days'] < 380)]
    if not annual.empty:
        annual = annual.copy()
        annual['dist'] = (annual['days'] - 365).abs()
        val = annual.sort_values('dist').iloc[0]
        return val['value'], val['source_filing_date']

    return None, None

def add_all_quarterly_metrics_from_df(facts_df: pd.DataFrame, report_date, metrics: dict):
    for my_name, gaap_tags in METRICS.items():
        if metrics.get(my_name) is not None:
            continue
        for tag in gaap_tags:
            val, _ = get_precise_quarterly_value(facts_df, tag, report_date)
            if val is not None:
                metrics[my_name] = val
                break
        # else:
        #     if metrics.get(my_name) is None:
        #         print(f"Pass 1 miss: {my_name} on {report_date}")  # temporary debug

def add_all_annual_metrics_from_df(facts_df: pd.DataFrame, report_date, metrics: dict):
    for my_name, gaap_tags in METRICS.items():
        if metrics.get(my_name) is not None:
            continue
        for tag in gaap_tags:
            val, _ = get_precise_annual_value(facts_df, tag, report_date)
            if val is not None:
                metrics[my_name] = val
                break

def apply_accounting_logic(metrics):
    # ── 0. Shares ──────────────────────────────────────────
    if metrics.get('commonStockSharesOutstanding') is None:
        shares_basic_and_dil = metrics.get('commonStockSharesOutstandingBasicAndDiluted')
        shares_dil = metrics.get('commonStockSharesOutstandingDiluted')
        if shares_basic_and_dil is not None and shares_dil is not None:
            metrics['commonStockSharesOutstanding'] = shares_basic_and_dil - shares_dil

    if metrics.get('commonStockSharesOutstandingDiluted') is None:
        shares_basic_and_dil = metrics.get('commonStockSharesOutstandingBasicAndDiluted')
        shares_basic = metrics.get('commonStockSharesOutstanding')
        if shares_basic_and_dil is not None and shares_basic is not None:
            metrics['commonStockSharesOutstandingDiluted'] = shares_basic_and_dil - shares_basic
            
    # ── 1. Balance sheet identities ──────────────────────────────────────────

    # Total Liabilities from the accounting equation
    if metrics.get('totalLiabilities') is None:
        total_sum = metrics.get('liabilitiesAndStockholdersEquity')
        equity    = metrics.get('totalShareholderEquity')
        if total_sum is not None and equity is not None:
            metrics['totalLiabilities'] = total_sum - equity

    # Shareholder equity from accounting equation (reverse direction)
    if metrics.get('totalShareholderEquity') is None:
        total_sum = metrics.get('liabilitiesAndStockholdersEquity')
        liab      = metrics.get('totalLiabilities')
        if total_sum is not None and liab is not None:
            metrics['totalShareholderEquity'] = total_sum - liab

    # Net working capital

    if metrics.get('totalCurrentAssets') is None:
        assets = metrics.get('totalAssets')
        nonc_assets = metrics.get('nonCurrentAssets')

        if assets is not None and nonc_assets is not None:
            metrics['totalCurrentAssets'] = assets - nonc_assets

    if metrics.get('totalCurrentLiabilities') is None:
        liab = metrics.get('totalLiabilities')
        nonc_liab = metrics.get('nonCurrentLiabilities')

        if liab is not None and nonc_liab is not None:
            metrics['totalCurrentLiabilities'] = liab - nonc_liab

    ca = metrics.get('totalCurrentAssets')
    cl = metrics.get('totalCurrentLiabilities')
    if ca is not None and cl is not None :
        metrics['netWorkingCapital'] = ca - cl

    # ── 2. Income statement derivations ─────────────────────────────────────

    # ── operatingExpenses = SGA + RD + COGS (last resort) ─────────────────
    if metrics.get('operatingExpenses') is None:
        sga  = metrics.get('sellingGeneralAndAdministrative')
        rd   = metrics.get('researchAndDevelopment')
        cogs = metrics.get('costofGoodsAndServicesSold')
        parts = [v for v in [sga, rd, cogs] if v is not None]
        if parts:
            metrics['operatingExpenses'] = sum(parts)

    # Operating Income fallback
    if metrics.get('operatingIncome') is None:
        gp = metrics.get('grossProfit')
        oe = metrics.get('operatingExpenses')
        if gp is not None and oe is not None:
            metrics['operatingIncome'] = gp - oe

    # ── incomeTaxExpense = current + deferred ──────────────────────────────
    if metrics.get('incomeTaxExpense') is None:
        current = metrics.get('currentIncomeTaxExpense')   # you'd need to scrape these
        deferred = metrics.get('deferredIncomeTaxExpense')  # as separate keys in METRICS
        if current is not None and deferred is not None:
            metrics['incomeTaxExpense'] = current + deferred
        elif current is not None:
            metrics['incomeTaxExpense'] = current
        elif deferred is not None:
            metrics['incomeTaxExpense'] = deferred

    if metrics.get('operatingIncome') is None:
        non_op = metrics.get('nonoperatingIncome')
        ni = metrics.get('netIncome')
        ite = metrics.get('incomeTaxExpense')
        ie = metrics.get('interestExpense')
        if all(v is not None for v in [non_op, ni, ite, ie]):
            metrics['operatingIncome'] = ni + ite + ie - non_op

    # D&A as sum of Depreciation and Ammortization
    if metrics.get('depreciationAndAmortization') is None:
        d = metrics.get('depreciation')
        am = metrics.get('amortization')

        if d is not None and am is not None:
            metrics['depreciationAndAmortization'] = d + am
        elif d is not None:
            metrics['depreciationAndAmortization'] = d
        elif am is not None:
            metrics['depreciationAndAmortization'] = am

    # COGS from Revenue - Gross Profit
    if metrics.get('costofGoodsAndServicesSold') is None:
        rev = metrics.get('totalRevenue')
        gp = metrics.get('grossProfit') 
        if rev is not None and gp is not None:
            metrics['costofGoodsAndServicesSold'] = rev - gp

    # Gross Profit from Revenue - COGS (reverse direction)
    if metrics.get('grossProfit') is None:
        rev  = metrics.get('totalRevenue')
        cogs = metrics.get('costofGoodsAndServicesSold')
        if rev is not None and cogs is not None:
            metrics['grossProfit'] = rev - cogs
    
    if metrics.get('grossProfit') is None:
        oe = metrics.get('operatingExpenses')
        oi = metrics.get('operatingIncome')
        if oe is not None and oi is not None:
            metrics['grossProfit'] = oe + oi
            
    # EBIT: Operating Income is effectively EBIT for most companies
    if metrics.get('ebit') is None:
        metrics['ebit'] = metrics.get('operatingIncome')

    # EBITDA = EBIT + D&A
    if metrics.get('ebitda') is None:
        ebit = metrics.get('ebit') or metrics.get('operatingIncome')
        da   = metrics.get('depreciationAndAmortization')
        if ebit is not None and da is not None:
            metrics['ebitda'] = ebit + da  

    # EBITDA fallback: Net Income + Tax + Interest + D&A
    if metrics.get('ebitda') is None:
        ni       = metrics.get('netIncome')
        tax      = metrics.get('incomeTaxExpense')
        interest = metrics.get('interestExpense')
        da       = metrics.get('depreciationAndAmortization')
        if all(v is not None for v in [ni, tax, interest, da]):
            metrics['ebitda'] = ni + tax + interest + da

    # EBT (Earnings Before Tax) = Net Income + Tax Expense
    if metrics.get('earningsBeforeTax') is None:
        ni  = metrics.get('netIncome')
        tax = metrics.get('incomeTaxExpense')
        if ni is not None and tax is not None:
            metrics['earningsBeforeTax'] = ni + tax
    
    # ── 3. Cash flow derivations ──────────────────────────────────────────────

    # Free Cash Flow = Operating CF - CapEx
    if metrics.get('freeCashFlow') is None:
        opf   = metrics.get('operatingCashflow')
        capex = metrics.get('capitalExpenditures')
        if opf is not None and capex is not None:
            # CapEx is usually stored as negative in filings; normalise
            metrics['freeCashFlow'] = opf - abs(capex)

    # ── 4. Debt & leverage aggregates ────────────────────────────────────────

    # Total debt
    if metrics.get('totalDebt') is None:
        st = metrics.get('shortTermDebt')
        lt = metrics.get('longTermDebt')
        if st is not None and lt is not None:
            metrics['totalDebt'] = st + lt
        elif lt is not None:
            metrics['totalDebt'] = lt # partial

    # Net debt = Total Debt - Cash
    if metrics.get('netDebt') is None:
        debt = metrics.get('totalDebt')
        cash = metrics.get('cashAndCashEquivalentsAtCarryingValue')
        if debt is not None and cash is not None:
            metrics['netDebt'] = debt - cash
    
    # ── 6. Market-based ratios (only if stock price available) ───────────────
    price  = metrics.get('stock_price')
    shares = metrics.get('commonStockSharesOutstanding')

    # Market Cap = Shares Count * Share Price
    if price and shares and shares > 0:
        market_cap = price * shares
        metrics['marketCap'] = market_cap

    # EV = Market Cap + Total Debt - Cash
    if metrics.get('enterpriseValue') is None:
        mc   = metrics.get('marketCap')
        debt = metrics.get('totalDebt')
        cash = metrics.get('cashAndCashEquivalentsAtCarryingValue')
        if mc is not None and debt is not None and cash is not None:
            metrics['enterpriseValue'] = mc + debt - cash
        elif mc is not None and debt is not None:
            metrics['enterpriseValue'] = mc + debt  # partial, no cash offset

    if metrics.get('dividendPayout') is None:
        dps = metrics.get('commonStockDividendsPerShareCashPaid')
        shares = metrics.get('commonStockSharesOutstanding')
        if dps is not None and shares is not None:
            metrics['dividendPayout'] = abs(dps * shares)


def compute_indicators(metrics: dict, prev_metrics: dict | None = None) -> dict: 
    """
        Step 2: compute all ratio-based indicators from enriched metrics.
        Requires apply_accounting_logic() to have been called first.
        Returns a flat dict of indicator values (None if inputs missing).
    """

    ind = {}

    def safe_div(a, b):
        if a is None or b is None or b == 0:
            return None
        else:
            return round(a / b, 6)
        
    rev   = metrics.get('totalRevenue')
    ni    = metrics.get('netIncome')
    gp    = metrics.get('grossProfit')
    cogs  = metrics.get('costofGoodsAndServicesSold')
    assets= metrics.get('totalAssets')
    eq    = metrics.get('totalShareholderEquity')
    liab  = metrics.get('totalLiabilities')
    ca    = metrics.get('totalCurrentAssets')
    cl    = metrics.get('totalCurrentLiabilities')
    ocf   = metrics.get('operatingCashflow')
    capex = metrics.get('capitalExpenditures')
    fcf   = metrics.get('freeCashFlow')
    ebit  = metrics.get('ebit')
    ebitda= metrics.get('ebitda')
    da    = metrics.get('depreciationAndAmortization')
    td    = metrics.get('totalDebt')
    nd    = metrics.get('netDebt')
    ev    = metrics.get('enterpriseValue')
    mc    = metrics.get('marketCap')
    price = metrics.get('stock_price')
    shares= metrics.get('commonStockSharesOutstanding')
    shares_d = metrics.get('commonStockSharesOutstandingDiluted')
    re    = metrics.get('retainedEarnings')
    div   = metrics.get('dividendPayout')
    nwc   = metrics.get('netWorkingCapital')
    interest = metrics.get('interestExpense')
    tax   = metrics.get('incomeTaxExpense')
    sbc   = metrics.get('stockBasedCompensation')

    # ── Profitability margins ────────────────────────────────────────────────
    ind['grossMargin']      = safe_div(gp, rev)
    ind['operatingMargin']  = safe_div(metrics.get('operatingIncome'), rev)
    ind['netProfitMargin']  = safe_div(ni, rev)
    ind['ebitdaMargin']     = safe_div(ebitda, rev)
    ind['fcfMargin']        = safe_div(fcf, rev)

    # ── Efficiency / returns ─────────────────────────────────────────────────
    ind['roe']  = safe_div(ni, eq)   # Return on Equity
    ind['roa']  = safe_div(ni, assets)  # Return on Assets
    ind['equityMultiplier'] = safe_div(assets, eq)

    # ROIC = EBIT*(1-tax_rate) / Invested Capital
    # Invested Capital = Total Equity + Total Debt - Cash
    cash = metrics.get('cashAndCashEquivalentsAtCarryingValue')
    invested_capital = None
    if eq is not None and td is not None and cash is not None:
        invested_capital = eq + td - cash
    elif eq is not None and td is not None:
        invested_capital = eq + td
    tax_rate = safe_div(tax, metrics.get('earningsBeforeTax')) if tax and metrics.get('earningsBeforeTax') else 0.21
    if ebit is not None and invested_capital:
        nopat = ebit * (1 - (tax_rate or 0.21))
        ind['roic'] = safe_div(nopat, invested_capital)
    else:
        ind['roic'] = None
    ind['investedCapital'] = invested_capital

    # ── Cash flow quality ────────────────────────────────────────────────────
    ind['cashKingRatio']  = safe_div(ocf, ni)    # OCF / Net Income (>1 = quality earnings)
    ind['ocfToRevenue']   = safe_div(ocf, rev)

    # SBC-adjusted FCF: removes non-cash compensation that dilutes shareholders
    # This is what most tech analysts mean by "real" FCF
    if fcf is not None and sbc is not None:
        ind['fcfExSbc'] = fcf - abs(sbc)
        ind['fcfExSbcMargin'] = safe_div(fcf - abs(sbc), rev)
    else:
        ind['fcfExSbc'] = None
        ind['fcfExSbcMargin'] = None

    # SBC as % of revenue — signals how much dilution the business is running
    ind['sbcToRevenue'] = safe_div(sbc, rev)

    # Owner Earnings = Net Income + D&A - Maintenance CapEx
    # Heuristic: maintenance CapEx ≈ D&A (conservative proxy when not reported separately)
    if ni is not None and da is not None and capex is not None:
        maint_capex = min(abs(capex), da)
        sbc_cost = abs(sbc) if sbc is not None else 0  # treat as 0 if not reported
        ind['ownerEarnings'] = ni + da - maint_capex - sbc_cost
    else:
        ind['ownerEarnings'] = None

    # Sloan Ratio = (Net Income - OCF) / Total Assets (accruals quality, <10% healthy)
    if ni is not None and ocf is not None and assets:
        ind['sloanRatio'] = safe_div(ni - ocf, assets)
    else:
        ind['sloanRatio'] = None

        # ── Liquidity & leverage ─────────────────────────────────────────────────
    ind['currentRatio']     = safe_div(ca, cl)
    ind['quickRatio']       = safe_div(ca - (metrics.get('inventory') or 0), cl)
    ind['debtToEquity']     = safe_div(td, eq)
    ind['debtToAssets']     = safe_div(td, assets)
    ind['interestCoverage'] = safe_div(ebit, interest)  # EBIT / Interest (>3 healthy)
    ind['netDebtToEbitda']  = safe_div(nd, ebitda)

    # ── Per share ────────────────────────────────────────────────────────────
    ind['eps']              = safe_div(ni, shares)
    ind['epsDiluted']       = safe_div(ni, shares_d)
    ind['bookValuePerShare']= safe_div(eq, shares)
    ind['fcfPerShare']      = safe_div(fcf, shares)
    ind['ocfPerShare']      = safe_div(ocf, shares)
    ind['revenuePerShare']  = safe_div(rev, shares)

    # ── Market multiples (need price) ────────────────────────────────────────
    if mc:
        ind['peRatio']      = safe_div(mc, ni)
        ind['pbRatio']      = safe_div(mc, eq)
        ind['psRatio']      = safe_div(mc, rev)
        ind['pfcfRatio']    = safe_div(mc, fcf)
        ind['pCfoRatio']    = safe_div(mc, ocf)
        ind['evToEbitda']   = safe_div(ev, ebitda)
        ind['evToFcf']      = safe_div(ev, fcf)
        ind['evToRevenue']  = safe_div(ev, rev)
        ind['evToEbit']     = safe_div(ev, ebit)
    else:
        for k in ['peRatio','pbRatio','psRatio','pfcfRatio','evToEbitda','evToRevenue','evToEbit']:
            ind[k] = None

    # ── Dividends ────────────────────────────────────────────────────────────
    # dividendPayout from XBRL is total cash paid; annualise from quarterly
    if div and price and shares and shares > 0:
        dps = abs(div) / shares                         # dividend per share this quarter
        ind['dividendYield']      = safe_div(dps * 4, price)   # annualised
    else:
        ind['dividendYield'] = None
    ind['dividendPayoutRatio']    = safe_div(abs(div) if div else None, ni)

    if prev_metrics:
        prev_rev = prev_metrics.get('totalRevenue')
        prev_eps = prev_metrics.get('eps') or safe_div(
            prev_metrics.get('netIncome'),
            prev_metrics.get('commonStockSharesOutstanding')
        )
        ind['revenueGrowthQoQ'] = safe_div(rev - prev_rev, abs(prev_rev)) if rev and prev_rev else None
        cur_eps = ind.get('eps')
        ind['epsGrowthQoQ']     = safe_div(cur_eps - prev_eps, abs(prev_eps)) if cur_eps and prev_eps else None

        # PEG Ratio = P/E / (EPS growth rate annualised)
        pe  = ind.get('peRatio')
        eps_g = ind.get('epsGrowthQoQ')
        if pe and eps_g and eps_g > 0:
            ind['pegRatio'] = safe_div(pe, eps_g * 4 * 100)  # growth as % annualised
        else:
            ind['pegRatio'] = None
    else:
        ind['revenueGrowthQoQ'] = None
        ind['epsGrowthQoQ']     = None
        ind['pegRatio']         = None

    # ── Altman Z-Score (public non-financial companies) ──────────────────────
    # Z = 1.2A + 1.4B + 3.3C + 0.6D + 1.0E
    if all(v is not None for v in [nwc, re, ebit, mc, liab, rev, assets]) and assets > 0:
        A = nwc  / assets           # Working Capital / Total Assets
        B = re   / assets           # Retained Earnings / Total Assets
        C = ebit / assets           # EBIT / Total Assets
        D = mc   / liab             # Market Cap / Total Liabilities
        E = rev  / assets           # Revenue / Total Assets
        ind['altmanZScore'] = round(1.2*A + 1.4*B + 3.3*C + 0.6*D + 1.0*E, 4)
        # ind['altmanA'] = round(A, 4)
        # ind['altmanB'] = round(B, 4)
        # ind['altmanC'] = round(C, 4)
        # ind['altmanD'] = round(D, 4)
        # ind['altmanE'] = round(E, 4)
    else:
        for k in ['altmanZScore','altmanA','altmanB','altmanC','altmanD','altmanE']:
            ind[k] = None

    # ── Rule of 40 (SaaS/growth metric) ─────────────────────────────────────
    rev_growth_pct = (ind.get('revenueGrowthQoQ') or 0) * 4 * 100 # annualised %
    fcf_margin_pct = (ind.get('fcfMargin') or 0) * 100
    if ind.get('revenueGrowthQoQ') is not None and ind.get('fcfMargin') is not None:
        ind['ruleOf40'] = round(rev_growth_pct + fcf_margin_pct, 2)
    else:
        ind['ruleOf40'] = None
    
    return ind

def sanitize_numeric(value):
    """Convert any value to float, or None if not possible."""
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value.replace(',', '').strip())
        except (ValueError, AttributeError):
            return None
    return None

def sanitize_metrics(metrics: dict) -> dict:
    """
    Run once after XBRL extraction and RENAME_TABLE mapping,
    before any accounting logic or indicator computation.
    Non-numeric fields are passed through as-is.
    """
    NON_NUMERIC = {'currency_symbol', 'fiscal_date_ending', 'filing_date', 'ticker'}
    return {
        k: (v if k in NON_NUMERIC else sanitize_numeric(v))
        for k, v in metrics.items()
    }

def parse_tenq_raw(filing, price_history: pd.Series, facts_df: pd.DataFrame) -> dict:
    tenq = filing.obj()
    raw_metrics = {}

    try:
        for old_name, value in tenq.financials.get_financial_metrics().items():
            if old_name in RENAME_TABLE:
                raw_metrics[RENAME_TABLE[old_name]] = value
    except Exception as e:
        print(f"Warning: financials extraction failed for 10-Q {filing.filing_date}: {e}")

    add_all_quarterly_metrics_from_df(facts_df, filing.period_of_report, raw_metrics)

    raw_metrics['fiscal_date_ending'] = tenq.period_of_report
    raw_metrics['filing_date'] = tenq.filing_date
    raw_metrics['stock_price'] = lookup_price(price_history, tenq.filing_date)
    raw_metrics['currency_symbol'] = tenq.financials.get_currency_symbol()
    raw_metrics['form_type'] = '10-Q'

    if raw_metrics.get('commonStockSharesOutstanding') is None:
        raw_metrics['commonStockSharesOutstanding'] = get_shares_from_cover_page_from_df(facts_df)

    return sanitize_metrics(raw_metrics)


def extract_annual_metrics(filing, price_history: pd.Series, facts_df: pd.DataFrame) -> dict:
    tenk = filing.obj()
    raw_metrics = {}

    try:
        for old_name, value in tenk.financials.get_financial_metrics().items():
            if old_name in RENAME_TABLE:
                raw_metrics[RENAME_TABLE[old_name]] = value
    except Exception as e:
        print(f"Warning: financials extraction failed for 10-K {filing.filing_date}: {e}")

    add_all_annual_metrics_from_df(facts_df, filing.period_of_report, raw_metrics)

    raw_metrics['fiscal_date_ending'] = tenk.period_of_report
    raw_metrics['filing_date'] = filing.filing_date
    raw_metrics['stock_price'] = lookup_price(price_history, filing.filing_date)
    raw_metrics['currency_symbol'] = tenk.financials.get_currency_symbol()

    if raw_metrics.get('commonStockSharesOutstanding') is None:
        raw_metrics['commonStockSharesOutstanding'] = get_shares_from_cover_page_from_df(facts_df)

    return sanitize_metrics(raw_metrics)


def get_shares_from_cover_page_from_df(facts_df: pd.DataFrame) -> float | None:
    """Same as get_shares_from_cover_page but reuses already-parsed facts_df."""
    try:
        cover = facts_df[
            (facts_df['clean_concept'] == 'EntityCommonStockSharesOutstanding') &
            (facts_df['period_start'].isna()) &
            (facts_df['dimension'].isna())
        ]
        if not cover.empty:
            return float(cover.iloc[-1]['value'])
    except Exception as e:
        print(f"Cover page shares lookup failed: {e}")
    return None

def prepare_facts_df(facts_df: pd.DataFrame) -> pd.DataFrame:
    """Pre-compute all derived columns once so lookups are just filters."""
    facts_df = facts_df.copy()
    if 'dimension' not in facts_df.columns:
        facts_df['dimension'] = None
    facts_df['clean_concept'] = facts_df['concept'].str.split(':|_').str[-1]

    # Normalise: use period_instant as period_end when period_end is missing
    if 'period_instant' in facts_df.columns:
        facts_df['period_end'] = facts_df['period_end'].fillna(facts_df['period_instant'])

    facts_df['period_end_ts'] = pd.to_datetime(facts_df['period_end'])
    facts_df['period_start_ts'] = pd.to_datetime(facts_df['period_start'])
    facts_df['days'] = (facts_df['period_end_ts'] - facts_df['period_start_ts']).dt.days
    return facts_df

FLOW_METRICS = {
    'totalRevenue', 'netIncome', 'grossProfit', 'operatingIncome',
    'operatingCashflow', 'capitalExpenditures', 'incomeTaxExpense',
    'interestExpense', 'depreciationAndAmortization', 'stockBasedCompensation',
    'operatingExpenses', 'nonoperatingIncome', 'researchAndDevelopment',
    'sellingGeneralAndAdministrative', 'comprehensiveIncome', 'freeCashFlow',
    'depreciation', 'amortization', 'dividendPayout', 'proceedsFromDebt', 
    'repaymentsOfDebt', 'proceedsFromEquityIssuance', 'sharesBuyback',
    'costofGoodsAndServicesSold', 'ebitda', 'currentIncomeTaxExpense', 
    'deferredIncomeTaxExpense'
}
STOCK_METRICS = {
    'totalAssets', 'totalLiabilities', 'totalShareholderEquity',
    'totalCurrentAssets', 'totalCurrentLiabilities', 'longTermDebt',
    'shortTermDebt', 'retainedEarnings', 'cashAndCashEquivalentsAtCarryingValue',
    'accountsReceivable', 'inventory', 'propertyPlantEquipmentNet', 'goodwill', 
    'intangibleAssets', 'operatingLeaseRightOfUseAsset', 'liabilitiesAndStockholdersEquity',
    'accountsPayable', 'accruedLiabilities', 'operatingLeaseLiability', 
    'noncontrollingInterest', 'deferredRevenue', 'longTermInvestments', 
}
SKIP_METRICS = {
    # These are weighted averages or per-share — don't subtract
    'commonStockSharesOutstanding',
    'commonStockSharesOutstandingDiluted',
    'commonStockDividendsPerShareCashPaid',
}

def derive_q4_metrics(annual_metrics: dict, q1: dict, q2: dict, q3: dict) -> dict:
    q4 = {}
    for metric in FLOW_METRICS:
        annual = annual_metrics.get(metric)
        prior = sum(filter(None, [q1.get(metric), q2.get(metric), q3.get(metric)]))
        q4[metric] = (annual - prior) if annual is not None else None

    for metric in STOCK_METRICS:
        q4[metric] = annual_metrics.get(metric) 

    for metric in SKIP_METRICS:
        q4[metric] = annual_metrics.get(metric)

    return q4
    
def extract_all_facts(filing, form_type: str) -> pd.DataFrame:
    xbrl = filing.xbrl()
    facts_df = xbrl.facts.to_dataframe()
    facts_df['source_filing_date'] = filing.filing_date
    facts_df['source_form_type'] = form_type
    return prepare_facts_df(facts_df)

def parse_full_history(ticker: str) -> pd.DataFrame | None:
    company = Company(ticker)
    price_history = get_price_history(ticker)
    
    filings_10q = [
        ("10-Q", f) for f in company.get_filings(form="10-Q", amendments=False)
        if f.filing_date >= date(2009, 1, 1)
    ]
    filings_10k = [
        ("10-K", f) for f in company.get_filings(form="10-K", amendments=False)
        if f.filing_date >= date(2009, 1, 1)
    ]

    if not filings_10q:
        print(f"Warning: No 10-Q filings found for {ticker}")
        return None
    
    all_filings = sorted(filings_10q + filings_10k, key=lambda x: x[1].period_of_report)

    # ── PASS 1: raw extraction only ──────────────────────────────────────────
    rows = []
    all_facts = []
    quarterly_buffer = []

    for form_type, filing in tqdm(all_filings, desc="Pass 1: extracting raw metrics"):
        try:
            facts_df = extract_all_facts(filing, form_type)
            all_facts.append(facts_df)

            if form_type == "10-Q":
                raw = parse_tenq_raw(filing, price_history, facts_df)
                rows.append(raw)
                quarterly_buffer.append(raw)

            elif form_type == "10-K":
                if len(quarterly_buffer) < 3:
                    print(f"Warning: Fewer than 3 quarters buffered before 10-K {filing.filing_date}, skipping Q4 derivation")
                    quarterly_buffer = []
                    continue

                annual_metrics = extract_annual_metrics(filing, price_history, facts_df)
                q1, q2, q3 = quarterly_buffer[-3], quarterly_buffer[-2], quarterly_buffer[-1]
                q4_raw = derive_q4_metrics(annual_metrics, q1, q2, q3)

                q4_raw['fiscal_date_ending'] = annual_metrics['fiscal_date_ending']
                q4_raw['filing_date'] = annual_metrics['filing_date']
                q4_raw['stock_price'] = annual_metrics['stock_price']
                q4_raw['currency_symbol'] = annual_metrics['currency_symbol']
                q4_raw['form_type'] = '10-K-derived'

                rows.append(q4_raw)
                quarterly_buffer = []
                
        except Exception as e:
                print(f"Skipping {form_type} filing {filing.filing_date}: {e}")
                continue
    
    if not rows:
        print(f"Warning: No data found for {ticker}")
        return None
    
    # ── PASS 2: backfill missing values from global facts pool ───────────────
    print("Pass 2: backfilling from later filings...")
    global_facts = pd.concat(all_facts, ignore_index=True)
    global_facts = prepare_facts_df(global_facts) 
    del all_facts

    global_facts_by_concept = {
        concept: group 
        for concept, group in global_facts.groupby('clean_concept')
    }

    for row in tqdm(rows, desc="Pass 2: backfilling"):
        report_date = str(row.get('fiscal_date_ending', ''))[:10]
        if not report_date:
            print("Warning: Missing report date, skipping")
            continue

        is_annual = row.get('form_type') == '10-K-derived'

        for my_name, gaap_tags in METRICS.items():
            if row.get(my_name) is not None:
                continue
            for tag in gaap_tags:
                tag_df = global_facts_by_concept.get(tag)
                if tag_df is None:
                    continue

                val, end_date_where_found = (
                    get_precise_annual_value(tag_df, tag, report_date)
                    if is_annual else
                    get_precise_quarterly_value(tag_df, tag, report_date)
                )
                if val is not None:
                    # filing_date = pd.to_datetime(end_date_where_found)
                    # period_date = pd.to_datetime(report_date)
                    # days_gap = (filing_date - period_date).days
                    # if days_gap > MAX_BACKFILL_DAYS:
                    #     continue
                    # print(f"Found tag: {my_name} for date {report_date} in statement reported at: {end_date_where_found}")
                    row[my_name] = sanitize_numeric(val)
                    break
            if my_name == "depreciationAndAmortization" and row.get(my_name) is None:
                print(f"Missing {my_name} for date {report_date}")
    del global_facts
    del global_facts_by_concept

    # ── PASS 3: accounting logic + indicators, once, in order ────────────────
    print("Pass 3: computing accounting logic and indicators...")

    prev_raw = None
    for row in tqdm(rows, desc="Pass 3: computing indicators"):
        apply_accounting_logic(row)

        indicators = compute_indicators(row, prev_raw)
        for k, v in indicators.items():
            row[f"ind_{k}"] = v
        
        prev_raw = {k: v for k, v in row.items() if not k.startswith('ind_')}

    # ── Build final DataFrame ─────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    del rows
    df['form_type'] = df['form_type'].fillna('10-Q')
    df.set_index("fiscal_date_ending", inplace=True)
    df['filing_date'] = pd.to_datetime(df['filing_date'])
    df.index = pd.to_datetime(df.index)

    return df

def save_tenq_history(ticker: str, df: pd.DataFrame):
    path = f"{EDGAR_DATA}/{ticker}.csv"
    os.makedirs(EDGAR_DATA, exist_ok=True)

    df_to_save = df.reset_index() if df.index.name else df
    df_to_save.to_csv(path, index=False)
    print(f"Success: Overwrote history for {ticker} at {path}")
    
def save_metadata(ticker: str, metadata: dict):
    path = f"{EDGAR_DATA}/metadata.json"
    os.makedirs(EDGAR_DATA, exist_ok=True)

    existing = {}
    if os.path.exists(path):
        with open(path) as f:
            existing = json.load(f)
    
    existing[ticker] = metadata
    with open(path, 'w') as f:
        json.dump(existing, f, indent=2)

def get_company_tickers():
    if os.path.exists('company_tickers.json'):
        with open('company_tickers.json', 'r') as f:
            data = json.load(f)
    else:
        r = requests.get(COMPANY_TICKERS_URL, headers=SEC_HEADERS)
        r.raise_for_status()
        data = r.json()
    return data


def data_pipeline():
    tickers_data = get_company_tickers()

    for entry in tickers_data.values():
        # ticker = entry["ticker"]
        start_time = time.time()
        ticker = "VTR"
        print(f"Parsing data for {entry["title"]} ({ticker})")

        metadata = get_company_metadata(ticker)
        if metadata["sector"] is None:
            print(f"Warning: Could not derive operating sector of: {entry["title"]} ({ticker}), skipping")
            continue

        metadata["title"] = entry["title"]
        save_metadata(ticker, metadata)

        df = parse_full_history(ticker)
        if df is not None:
            save_tenq_history(ticker, df)
            print(f"Successfully finished parsing {entry["title"]} ({ticker}) data, rows: {len(df)}")
            print(f"Total time: {round(time.time() - start_time, 4)}s")
        else:
            print(f"DataFrame for {ticker} is None, skipping")
            continue

        
        break

def export_for_training() -> pd.DataFrame:
    with open(f"{EDGAR_DATA}/metadata.json", "r") as f:
        metadata = json.load(f)

    frames = []
    for ticker, meta in metadata.items():
        path = f"{EDGAR_DATA}/{ticker}.csv"
        if os.path.exists(path):
            base = pd.read_csv(path)

            extra = pd.DataFrame({
                "ticker": ticker,
                "sector": meta.get("sector"),
            }, index=base.index)

            df = pd.concat([base, extra], axis=1)
            frames.append(df)

    if frames:
        return pd.concat(frames, ignore_index=True)
    else:
        print("Warnign: No frames found, returning empty DF")
        return pd.DataFrame()

data_pipeline()

Parsing data for NVIDIA CORP (VTR)


In [ ]:
pd.set_option('display.max_rows', None)

df = export_for_training()
df = df[df['ticker'] == 'VTR']
df.tail()
print(len(df))
df.isna().sum()

64


fiscal_date_ending                        0
totalRevenue                              0
operatingIncome                           0
netIncome                                 0
totalAssets                               0
totalLiabilities                          0
totalShareholderEquity                    0
totalCurrentAssets                        0
totalCurrentLiabilities                   0
operatingCashflow                         0
capitalExpenditures                       0
freeCashFlow                              0
commonStockSharesOutstanding              0
commonStockSharesOutstandingDiluted       0
operatingExpenses                         0
nonoperatingIncome                        0
liabilitiesAndStockholdersEquity          0
grossProfit                               0
costofGoodsAndServicesSold               47
depreciation                             64
amortization                             45
depreciationAndAmortization              34
cashAndCashEquivalentsAtCarrying

In [6]:

ticker = "PEG"

company = Company(ticker)
filings = company.get_filings(form="10-K")

# filing = None
# for f in filings:
#     print(f.period_of_report)
#     if f.period_of_report == "2024-09-30":
#         filing = f
#         break
    
filing = filings[10]
xbrl = filing.xbrl()

facts = xbrl.facts.to_dataframe().to_json('out.json', indent=2)



TODO:
- List of key fundamentals (ones that are actually used to count metrics)
- Filter out data points that are missing a lot of those key fundamentals
- List of values for which it is valid to use interpolation to fill missing value
- At the end of pipeline, if big percentage of indicators are missing, get rid of this data (company) completely
- How to deal with missing quarters (explicitly removed) when training models in sliding windows (~5 training), predict whether price change will beat markete change during next year?
- Collect dataframe with whole market price (S&P ?) for years 2009 - Present (for comparison)